<a href="https://colab.research.google.com/github/BF667/notebook/blob/main/ShareGPT_webdataset_maker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1) Install dependencies

This cell installs the libraries required to fetch and parse web pages. Use the checkbox to skip installation if the packages are already present in your environment.

In [ ]:
#@title Install dependencies (requests, beautifulsoup4, tqdm)
#@markdown If you're running this in Colab or a fresh environment, enable installation. If you already have the packages installed, set install_deps to False.
install_deps = True  #@param {type:"boolean"}

if install_deps:
    import sys
    !{sys.executable} -m pip install --quiet requests beautifulsoup4 tqdm
    print('Dependencies installed.')
else:
    print('Skipping installation. Make sure requests & beautifulsoup4 are available.')

## 2) Configuration and tutorial notes

Below are configurable options for fetching and parsing. They appear as form fields in Colab (or as comments in other environments). Adjust them before running the extractor.

Explanation of key options:
- timeout_seconds: maximum seconds to wait for a single HTTP request.
- max_retries: number of automatic retry attempts for transient network errors.
- user_agent: identifies the HTTP client. Some sites block non-browser user agents.
- selectors: a prioritized list of CSS selectors (or tags) to try for extracting main text; order matters.
- output_filename: the JSONL filename written to disk.


In [ ]:
#@title Fetch & parsing configuration
#@markdown Adjust these parameters to control fetch behavior and extraction strategy.
timeout_seconds = 10  #@param {type:"number"}
max_retries = 2  #@param {type:"integer"}
user_agent = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"  #@param {type:"string"}

#@markdown Provide selectors (one per line) to try in order. Common choices: 'main', 'article', 'div#content', 'div.post', 'body'.
selectors_text = "main\narticle\ndiv#content\ndiv.post\nbody"  #@param {type:"string"}
selectors = [s.strip() for s in selectors_text.splitlines() if s.strip()]

#@markdown Output filename for the generated ShareGPT-format JSONL dataset.
output_filename = "sharegpt_dataset.jsonl"  #@param {type:"string"}

print('Configuration:')
print(f' timeout_seconds = {timeout_seconds}')
print(f' max_retries = {max_retries}')
print(f' user_agent = {user_agent[:60]}...')
print(f' selectors = {selectors}')
print(f' output_filename = {output_filename}')

## 3) Fetch and parse function

This function performs HTTP GET with retries and extracts text using BeautifulSoup. It tries the ordered selectors first and falls back to a lightweight heuristic if none found.
The function returns a dictionary with metadata for easier debugging: url, success flag, status_code (if available), extracted_text, and error message (if any).

In [ ]:
#@title Define fetch_and_parse(url)
#@markdown The function below uses the configuration variables defined earlier (timeout_seconds, max_retries, user_agent, selectors).
import time
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

def fetch_and_parse(url):
    """
    Fetches a URL with retries and extracts main textual content.
    Returns a dict with keys: url, success (bool), status_code (int|None), text (str|None), error (str|None).
    """
    headers = {"User-Agent": user_agent}
    session = requests.Session()
    retries = Retry(total=max_retries, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])
    session.mount('http://', HTTPAdapter(max_retries=retries))
    session.mount('https://', HTTPAdapter(max_retries=retries))

    try:
        resp = session.get(url, headers=headers, timeout=timeout_seconds)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        return {"url": url, "success": False, "status_code": getattr(e.response, 'status_code', None), "text": None, "error": str(e)}

    try:
        soup = BeautifulSoup(resp.content, 'html.parser')

        # Try the configured selectors first
        for sel in selectors:
            # If selector looks like a simple tag name (no special chars), use find by name
            try:
                if sel.startswith('body') or sel.startswith('div') or sel.startswith('article') or sel.startswith('main') or '#' in sel or '.' in sel:
                    element = soup.select_one(sel)
                else:
                    element = soup.find(sel)
            except Exception:
                element = None

            if element:
                text = element.get_text(separator='\n', strip=True)
                if text:
                    return {"url": url, "success": True, "status_code": resp.status_code, "text": text, "error": None}

        # Fallback: heuristics - choose the longest text node among paragraphs and headings
        paragraphs = [p.get_text(separator=' ', strip=True) for p in soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])]
        if paragraphs:
            # Join the largest continuous block by length
            joined = '\n\n'.join(paragraphs)
            if len(joined) > 50:
                return {"url": url, "success": True, "status_code": resp.status_code, "text": joined, "error": None}

        # As a last resort, return full page text (may include navigation, footer, etc.)
        full_text = soup.get_text(separator='\n', strip=True)
        return {"url": url, "success": True, "status_code": resp.status_code, "text": full_text, "error": None}

    except Exception as e:
        return {"url": url, "success": False, "status_code": resp.status_code if 'resp' in locals() else None, "text": None, "error": str(e)}

print('fetch_and_parse defined.')

## 4) Provide URLs list

You can provide URLs in the multi-line text box. Each new line is treated as one URL. Use real URLs for production; the sample ones below are placeholders.

In [ ]:
#@title URLs to fetch (one per line)
#@markdown Provide the list of pages you want to include in the dataset. Example:
#@markdown https://example.com/article1
#@markdown https://example.com/article2
urls_text = """

https://www.example.com
https://www.anothersite.org
https://www.yetanothersite.net"

"""
urls = [u.strip() for u in urls_text.splitlines() if u.strip()]
print(f'Loaded {len(urls)} URLs.')
for u in urls[:10]:
    print('-', u)

## 5) Extract text from each URL

This step loops over the URLs and uses fetch_and_parse to collect content. A progress bar is shown so you can monitor progress.
The results list contains dictionaries with metadata for each fetch. Inspect failed entries to understand issues (timeouts, 404, blocked requests, etc.).

In [ ]:
#@title Run extraction
#@markdown Click Run to start fetching and parsing the URLs. You can interrupt if needed.
from tqdm import tqdm

results = []
for url in tqdm(urls, desc='Processing URLs'):
    info = fetch_and_parse(url)
    results.append(info)

successes = sum(1 for r in results if r['success'])
print(f'Done. Successful extractions: {successes} / {len(results)}')

# Show a short report of failures (if any)
failures = [r for r in results if not r['success']]
if failures:
    print('\nFailures (first 10):')
    for f in failures[:10]:
        print('-', f['url'], '| status:', f.get('status_code'), '| error:', f.get('error'))
else:
    print('No failures reported.')

## 6) Preview extracted content

Inspect the first few extracted pages to ensure the content looks reasonable before saving.
If the extraction picked up navigation or unrelated text, adjust the selectors in the configuration cell and re-run the extraction.

In [ ]:
#@title Preview first successful extraction
#@markdown This prints the URL and the first ~1000 characters of the extracted text for the first successful fetch.
preview_count = 1  #@param {type:"integer"}
shown = 0
for r in results:
    if r['success']:
        print('URL:', r['url'])
        text = r['text'] or ''
        print('\n--- Extracted snippet ---\n')
        print(text[:1000])
        print('\n--- end snippet ---\n')
        shown += 1
        if shown >= preview_count:
            break
if shown == 0:
    print('No successful extractions to preview.')

## 7) Format as ShareGPT JSONL and save

By default each entry is formatted as a conversation with two messages:
- human: the prompt to perform (e.g., 'Please summarize the following text:')
- gpt: the extracted document text

You can modify the prompt text below if you prefer different instructions (e.g., 'Extract main points', 'Create Q&A pairs', etc.).

In [ ]:
#@title Save ShareGPT JSONL
#@markdown Adjust prompt_text if you want the 'human' message to ask for something else (summary, bullet points, Q&A generation, etc.).
prompt_text = "Please summarize the following text:"  #@param {type:"string"}
save_only_successful = True  #@param {type:"boolean"}

import json

to_save = []
for r in results:
    if save_only_successful and not r['success']:
        continue
    text = r['text'] or ''
    entry = {
        "source_url": r['url'],
        "conversations": [
            {"from": "human", "value": prompt_text},
            {"from": "gpt", "value": text}
        ]
    }
    to_save.append(entry)

with open(output_filename, 'w', encoding='utf-8') as f:
    for e in to_save:
        f.write(json.dumps(e, ensure_ascii=False) + '\n')

print(f'Saved {len(to_save)} entries to {output_filename}')

## 8) Download the dataset (Colab)

If you're running this in Colab you can download the file with the snippet below. In other environments the file will be available on the filesystem of the runtime.


In [ ]:
#@title (Optional) Download the generated JSONL file (Colab)
#@markdown Run this cell in Colab to get a download link. In non-Colab environments, skip this.
try:
    from google.colab import files  # type: ignore
    files.download(output_filename)
except Exception as e:
    print('Download helper failed or not running in Colab:', e)
    print('File saved at:', output_filename)

## Final notes and troubleshooting

- If pages are blocked (403) or return unexpected HTML (very short or none), try changing the user_agent to a modern browser UA string.
- If the extraction pulls nav/footer content rather than body text, add more specific selectors (e.g., 'div.article-content', 'div#main-article') to the selectors list and re-run extraction.
- For large numbers of URLs consider batching, rate-limiting, or using a headless browser approach (Selenium, Playwright) for JavaScript-heavy pages.
- Respect robots.txt and terms of service for target websites. This tool is for authorized scraping or publicly available content only.

If you'd like, I can further enhance this notebook to:
- Produce multiple ShareGPT conversation templates (e.g., summarization, Q&A, extraction of entities).
- Add parallel fetching with safe rate-limiting.
- Integrate readability libraries (newspaper3k) for improved extraction.

Happy dataset-building!